# Coleta de Indicadores Macroeconômicos

Este notebook realiza a coleta de indicadores macroeconômicos utilizados no projeto:

- Dólar/Real - PTAX de venda
- Meta da Taxa Selic
- IPCA

O período de análise compreende janeiro de 2021 a dezembro de 2025.

Os dados são obtidos a partir de fontes oficiais do Banco Central do Brasil e posteriormente padronizados para utilização nas etapas de modelagem e análise.

In [0]:
import requests
import pandas as pd

## Dólar/Real

A cotação do dólar é obtida por meio do serviço PTAX do Banco Central do Brasil. Para o projeto será utilizada a cotação de venda do boletim de fechamento.

In [0]:
url_dolar = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/"
    "versao/v1/odata/CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params_dolar = {
    "@dataInicial": "'01-01-2021'",
    "@dataFinalCotacao": "'12-31-2025'",
    "$format": "json"
}

response_dolar = requests.get(
    url_dolar,
    params=params_dolar,
    timeout=60
)

response_dolar.raise_for_status()

dados_dolar = response_dolar.json()["value"]

df_dolar = pd.DataFrame(dados_dolar)

display(df_dolar.head())

In [0]:
df_dolar = df_dolar[
    [
        "dataHoraCotacao",
        "cotacaoCompra",
        "cotacaoVenda"
    ]
].copy()

df_dolar["data"] = pd.to_datetime(
    df_dolar["dataHoraCotacao"]
).dt.normalize()

df_dolar = (
    df_dolar
    .sort_values("dataHoraCotacao")
    .groupby("data", as_index=False)
    .last()
)

df_dolar = df_dolar[
    ["data", "cotacaoCompra", "cotacaoVenda"]
]

df_dolar["indicador"] = "USD_BRL"

display(df_dolar.head())

## Meta Selic

A Meta Selic é obtida por meio do Sistema Gerenciador de Séries Temporais (SGS) do Banco Central do Brasil.

A série utilizada é a 432, que representa a meta para a taxa Selic definida pelo Copom, expressa em percentual ao ano.

In [0]:
url_selic = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
    "?formato=json"
    "&dataInicial=01/01/2021"
    "&dataFinal=31/12/2025"
)

response_selic = requests.get(
    url_selic,
    timeout=60
)

response_selic.raise_for_status()

dados_selic = response_selic.json()

df_selic = pd.DataFrame(dados_selic)

df_selic["data"] = pd.to_datetime(
    df_selic["data"],
    format="%d/%m/%Y"
)

df_selic["selic_meta_pct_aa"] = (
    df_selic["valor"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_selic = df_selic[
    ["data", "selic_meta_pct_aa"]
]

df_selic["indicador"] = "SELIC_META"

display(df_selic.head())

## IPCA

O IPCA é utilizado como medida de inflação no projeto.

A série é mensal e será utilizada posteriormente para cálculo de retorno real e análise dos ativos em diferentes cenários inflacionários.

In [0]:
url_ipca = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
    "?formato=json"
    "&dataInicial=01/01/2021"
    "&dataFinal=31/12/2025"
)

response_ipca = requests.get(
    url_ipca,
    timeout=60
)

response_ipca.raise_for_status()

dados_ipca = response_ipca.json()

df_ipca = pd.DataFrame(dados_ipca)

df_ipca["data"] = pd.to_datetime(
    df_ipca["data"],
    format="%d/%m/%Y"
)

df_ipca["ipca_pct_mes"] = (
    df_ipca["valor"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_ipca = df_ipca[
    ["data", "ipca_pct_mes"]
]

df_ipca["indicador"] = "IPCA"

display(df_ipca.head())

## Validação da qualidade dos dados

Nesta etapa são verificadas a quantidade de registros, o período disponível, a presença de valores nulos e possíveis duplicidades.

In [0]:
print("=== DÓLAR ===")
print("Registros:", len(df_dolar))
print("Período:", df_dolar["data"].min(), "até", df_dolar["data"].max())
print("Nulos:")
print(df_dolar.isnull().sum())
print("Duplicados:", df_dolar.duplicated(subset=["data"]).sum())

print("\n=== SELIC ===")
print("Registros:", len(df_selic))
print("Período:", df_selic["data"].min(), "até", df_selic["data"].max())
print("Nulos:")
print(df_selic.isnull().sum())
print("Duplicados:", df_selic.duplicated(subset=["data"]).sum())

print("\n=== IPCA ===")
print("Registros:", len(df_ipca))
print("Período:", df_ipca["data"].min(), "até", df_ipca["data"].max())
print("Nulos:")
print(df_ipca.isnull().sum())
print("Duplicados:", df_ipca.duplicated(subset=["data"]).sum())

## Resultado da coleta

Os três indicadores macroeconômicos foram coletados para o período definido no projeto, sem ocorrência de valores nulos ou registros duplicados.

As diferenças na quantidade de registros são decorrentes das características de cada série: a cotação do dólar está disponível em dias de cotação, a Meta Selic possui frequência diária e o IPCA possui frequência mensal.

A compatibilização das diferentes granularidades será realizada posteriormente na etapa de tratamento e modelagem dos dados.